#### Chapter4 Homework：
#### 1、使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少

In [2]:
from datasets import load_dataset
dataset = load_dataset("yelp_review_full")
dataset

/root/miniconda3/envs/transformers/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [3]:
from transformers import AutoTokenizer

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [6]:
#small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
#small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))
# 全量时间太久（约42个小时）
middle_train_dataset = tokenized_datasets["train"].shuffle(seed=42)
middle_eval_dataset = tokenized_datasets["test"].shuffle(seed=42)

### 加载模型权重

In [7]:
from transformers import AutoModelForSequenceClassification

In [8]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 训练评估指标

In [9]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### 训练超参数

In [11]:
from transformers import TrainingArguments, Trainer

In [12]:
model_dir = "models/bert-base-cased-finetune-yelp-small"
training_args = TrainingArguments(output_dir=model_dir,
                               evaluation_strategy="epoch",
                               per_device_train_batch_size=48, 
                               num_train_epochs=3,
                               logging_steps=100)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=middle_train_dataset,
    eval_dataset=middle_eval_dataset,
    compute_metrics=compute_metrics,
)

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [14]:
#trainer.train()

In [15]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(2000))

In [16]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 1.6612197160720825,
 'eval_accuracy': 0.2125,
 'eval_runtime': 23.5581,
 'eval_samples_per_second': 84.897,
 'eval_steps_per_second': 10.612}

In [17]:
trainer.save_model(model_dir)

In [18]:
trainer.save_state()